In [ ]:
!pip install faker -q

import sqlite3
import random
from datetime import datetime, timedelta
from faker import Faker

fake = Faker('en_IN')
random.seed(42)

DB_PATH = "/content/phc_agent.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
print("DB initialized at", DB_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.0 MB/s eta 0:00:00
DB initialized at /content/phc_agent.db


In [ ]:
cur.executescript("""
DROP TABLE IF EXISTS medicine_stock;
DROP TABLE IF EXISTS patients;
DROP TABLE IF EXISTS visits;
DROP TABLE IF EXISTS staff_availability;
DROP TABLE IF EXISTS alerts;

CREATE TABLE medicine_stock (
    med_id INTEGER PRIMARY KEY AUTOINCREMENT,
    med_name TEXT NOT NULL,
    category TEXT,
    current_qty INTEGER,
    reorder_threshold INTEGER,
    unit TEXT,
    last_restocked DATE,
    expiry_date DATE
);

CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    age INTEGER,
    gender TEXT,
    village TEXT,
    phone TEXT
);

CREATE TABLE visits (
    visit_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id INTEGER,
    visit_date DATE,
    visit_time TEXT,
    symptom_category TEXT,
    diagnosis TEXT,
    medicines_prescribed TEXT,
    FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
);

CREATE TABLE staff_availability (
    staff_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    role TEXT,
    shift_date DATE,
    shift_start TEXT,
    shift_end TEXT,
    is_present INTEGER
);

CREATE TABLE alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    alert_type TEXT,
    severity TEXT,
    message TEXT,
    triggered_at TIMESTAMP,
    sent_to_municipal INTEGER DEFAULT 0
);
""")
conn.commit()
print("Schema created.")

Schema created.


In [ ]:
#Cell 3: Synthetic medicine stock

MEDICINES = [
    ("Paracetamol 500mg", "Analgesic", "tablets"),
    ("ORS Packets", "Hydration", "packets"),
    ("Amoxicillin 250mg", "Antibiotic", "capsules"),
    ("Iron Folic Acid", "Supplement", "tablets"),
    ("ORS + Zinc", "Hydration", "kits"),
    ("Ibuprofen 400mg", "Analgesic", "tablets"),
    ("Cough Syrup", "Respiratory", "bottles"),
    ("Insulin", "Diabetes", "vials"),
    ("BP Medication (Amlodipine)", "Cardiac", "tablets"),
    ("Antiseptic Solution", "First Aid", "bottles"),
    ("Vitamin D3", "Supplement", "tablets"),
    ("Measles Vaccine", "Vaccine", "vials"),
]

for name, cat, unit in MEDICINES:
    qty = random.randint(5, 500)
    threshold = int(qty * random.uniform(0.15, 0.3)) if qty > 20 else 10
    restocked = fake.date_between(start_date="-60d", end_date="today")
    expiry = fake.date_between(start_date="+30d", end_date="+2y")
    cur.execute("""
        INSERT INTO medicine_stock (med_name, category, current_qty, reorder_threshold, unit, last_restocked, expiry_date)
        VALUES (?,?,?,?,?,?,?)
    """, (name, cat, qty, threshold, unit, restocked, expiry))

conn.commit()
print(f"{len(MEDICINES)} medicines seeded.")

12 medicines seeded.


/tmp/ipykernel_16572/2018835027.py:23: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cur.execute("""


 Synthetic patients + visits (footfall pattern)


In [ ]:
SYMPTOMS = ["Fever", "Diarrhea", "Respiratory Infection", "Hypertension",
            "Diabetes Checkup", "Injury/First Aid", "Antenatal Checkup",
            "Vaccination", "Skin Infection", "General Checkup"]

VILLAGES = [fake.city() for _ in range(6)]

# Generate 90 days of visit history with realistic patterns:
# - Higher footfall on Mon/Tue (post-weekend)
# - Seasonal spike simulated in one month (monsoon -> fever/diarrhea spike)
today = datetime.today()
patient_ids = []

for i in range(500):
    cur.execute("INSERT INTO patients (name, age, gender, village, phone) VALUES (?,?,?,?,?)",
                (fake.name(), random.randint(1, 85), random.choice(["M", "F"]),
                 random.choice(VILLAGES), fake.phone_number()))
    patient_ids.append(cur.lastrowid)

for day_offset in range(90, 0, -1):
    visit_date = today - timedelta(days=day_offset)
    weekday = visit_date.weekday()  # 0=Mon
    is_monsoon = visit_date.month in [7, 8, 9]

    base_footfall = 15 if weekday < 5 else 8
    if weekday in [0, 1]:
        base_footfall += 6
    if is_monsoon:
        base_footfall += random.randint(5, 15)

    daily_footfall = max(3, int(random.gauss(base_footfall, 4)))

    for _ in range(daily_footfall):
        pid = random.choice(patient_ids)
        symptom = random.choices(
            SYMPTOMS,
            weights=[3 if is_monsoon and s in ["Fever", "Diarrhea"] else 1 for s in SYMPTOMS]
        )[0]
        visit_time = f"{random.randint(9,16):02d}:{random.choice(['00','15','30','45'])}"
        cur.execute("""
            INSERT INTO visits (patient_id, visit_date, visit_time, symptom_category, diagnosis, medicines_prescribed)
            VALUES (?,?,?,?,?,?)
        """, (pid, visit_date.date(), visit_time, symptom, symptom,
              random.choice([m[0] for m in MEDICINES])))

conn.commit()
print("Patients + 90 days of visit history generated.")

Patients + 90 days of visit history generated.


/tmp/ipykernel_16572/1088887847.py:39: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cur.execute("""


Cell 5: Staff availability


In [ ]:
STAFF = [
    ("Dr. Sharma", "Medical Officer"),
    ("Dr. Iyer", "Medical Officer"),
    ("Nurse Kavita", "Staff Nurse"),
    ("Nurse Reema", "Staff Nurse"),
    ("Pharmacist Vinod", "Pharmacist"),
    ("ANM Sunita", "ANM"),
]

for name, role in STAFF:
    for day_offset in range(7):
        shift_date = (today + timedelta(days=day_offset)).date()
        present = random.choices([1, 0], weights=[0.9, 0.1])[0]
        cur.execute("""
            INSERT INTO staff_availability (name, role, shift_date, shift_start, shift_end, is_present)
            VALUES (?,?,?,?,?,?)
        """, (name, role, shift_date, "09:00", "17:00", present))

conn.commit()
print("Staff roster generated for next 7 days.")

Staff roster generated for next 7 days.


/tmp/ipykernel_16572/3902190562.py:14: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cur.execute("""


Cell 6: Quick sanity check

In [ ]:
for table in ["medicine_stock", "patients", "visits", "staff_availability"]:
    count = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

print("\nSample low-stock items:")
low = cur.execute("SELECT med_name, current_qty, reorder_threshold FROM medicine_stock WHERE current_qty < reorder_threshold").fetchall()
for row in low:
    print(row)

medicine_stock: 12 rows
patients: 500 rows
visits: 1291 rows
staff_availability: 42 rows

Sample low-stock items:


Cell 7: Config (using Colab Secrets)

In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.7 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq

# Add your key in Colab Secrets (🔑 icon) with name: GROQ_API_KEY
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key="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")

MODEL = "llama-3.3-70b-versatile"

In [ ]:
!pip install groq -q

In [ ]:
import json

def check_stock(med_name=None):
    """Check medicine stock. If med_name is None, returns all low-stock items."""
    if med_name:
        row = cur.execute(
            "SELECT med_name, current_qty, reorder_threshold, unit, expiry_date FROM medicine_stock WHERE med_name LIKE ?",
            (f"%{med_name}%",)
        ).fetchone()
        if not row:
            return {"error": f"No medicine found matching '{med_name}'"}
        return {"med_name": row[0], "current_qty": row[1], "reorder_threshold": row[2],
                "unit": row[3], "expiry_date": row[4], "status": "LOW" if row[1] < row[2] else "OK"}
    else:
        rows = cur.execute(
            "SELECT med_name, current_qty, reorder_threshold, unit FROM medicine_stock WHERE current_qty < reorder_threshold"
        ).fetchall()
        return {"low_stock_items": [{"med_name": r[0], "current_qty": r[1], "threshold": r[2], "unit": r[3]} for r in rows]}


def predict_footfall(days_ahead=1):
    """Predict footfall using recent trend (simple moving average + weekday pattern)."""
    rows = cur.execute("""
        SELECT visit_date, COUNT(*) as cnt FROM visits
        GROUP BY visit_date ORDER BY visit_date DESC LIMIT 30
    """).fetchall()
    if not rows:
        return {"error": "No visit history available"}

    avg_footfall = sum(r[1] for r in rows) / len(rows)
    target_date = datetime.today() + timedelta(days=int(days_ahead))
    weekday = target_date.weekday()

    weekday_boost = 1.3 if weekday in [0, 1] else (0.7 if weekday >= 5 else 1.0)
    predicted = round(avg_footfall * weekday_boost)

    return {
        "target_date": target_date.strftime("%Y-%m-%d"),
        "predicted_footfall": predicted,
        "based_on_avg": round(avg_footfall, 1),
        "note": "Simple trend-based estimate; replace with time-series model for production"
    }


def check_availability(date=None):
    """Check staff/doctor availability for a given date (default: today)."""
    target_date = date or datetime.today().strftime("%Y-%m-%d")
    rows = cur.execute(
        "SELECT name, role, is_present FROM staff_availability WHERE shift_date = ?",
        (target_date,)
    ).fetchall()
    if not rows:
        return {"error": f"No roster data for {target_date}"}
    present = [{"name": r[0], "role": r[1]} for r in rows if r[2] == 1]
    absent = [{"name": r[0], "role": r[1]} for r in rows if r[2] == 0]
    return {"date": target_date, "present_staff": present, "absent_staff": absent,
             "doctors_available": sum(1 for p in present if "Medical Officer" in p["role"])}


def raise_municipal_alert(alert_type, severity, message):
    """Log an alert and send email to municipal authority."""
    cur.execute(
        "INSERT INTO alerts (alert_type, severity, message, triggered_at) VALUES (?,?,?,?)",
        (alert_type, severity, message, datetime.now())
    )
    conn.commit()
    email_sent = send_alert_email(alert_type, severity, message)
    return {"logged": True, "email_sent": email_sent, "alert_type": alert_type, "severity": severity}

In [37]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key="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")

MODEL = "llama-3.3-70b-versatile"

In [40]:
print(run_agent("What medicines are running low?"))
print(run_agent("What's the predicted footfall for tomorrow?"))
print(run_agent("Who's available today?"))

No medicines are currently running low. Our stock levels are above the 10% threshold for all items.
The predicted footfall for tomorrow, 2026-07-05, is 10. This estimate is based on a simple average of 14.5 and should be replaced with a time-series model for more accurate production predictions.
I don't have the roster data for today. Please provide more information or check the staff roster manually.


In [39]:
TOOLS = [
    {"type": "function", "function": {"name": "check_stock", "description": "Check medicine stock levels. Omit med_name to get all low-stock items.",
        "parameters": {"type": "object", "properties": {"med_name": {"type": "string"}}}}},
    {"type": "function", "function": {"name": "predict_footfall", "description": "Predict patient footfall for a future date.",
        "parameters": {"type": "object", "properties": {"days_ahead": {"type": "integer"}}}}},
    {"type": "function", "function": {"name": "check_availability", "description": "Check doctor/staff availability for a date (YYYY-MM-DD).",
        "parameters": {"type": "object", "properties": {"date": {"type": "string"}}}}},
    {"type": "function", "function": {"name": "raise_municipal_alert", "description": "Raise an alert to municipal authority via email.",
        "parameters": {"type": "object", "properties": {
            "alert_type": {"type": "string"}, "severity": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
            "message": {"type": "string"}}, "required": ["alert_type", "severity", "message"]}}},
]

AVAILABLE_FUNCTIONS = {
    "check_stock": check_stock,
    "predict_footfall": predict_footfall,
    "check_availability": check_availability,
    "raise_municipal_alert": raise_municipal_alert,
}

def run_agent(user_query):
    messages = [
        {"role": "system", "content": "You are a PHC management assistant. Use tools to answer queries about stock, footfall, staff availability, and raise alerts when thresholds are critical (e.g. stock below 10% or footfall 2x normal). Be concise and factual."},
        {"role": "user", "content": user_query}
    ]

    response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto")
    msg = response.choices[0].message

    if msg.tool_calls:
        messages.append(msg)
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            result = AVAILABLE_FUNCTIONS[fn_name](**fn_args)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})

        final = client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content
    return msg.content